# 05B — Temporal / Context Feature Extraction

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

Stage 05A is now complete enough to proceed.

## Stage-05A result used here

The final static table contains:
- 200 ML-eligible landslide positives
- 3,000 background candidates
- static validity flags
- terrain
- SoilGrids
- major-fault distance
- structured JMG geology

Some samples remain invalid for models requiring soil. They are **not silently filled**.

---

# Purpose of Stage 05B

This notebook prepares a **date-matched temporal case-control dataset** and extracts time-sensitive/context predictors.

## Temporal design

For each dated, static-valid landslide:

- `reference_date = event_date`
- select 2 static-valid background controls
- assign the same `reference_date` to those controls

This makes rainfall / soil-moisture comparison temporally meaningful.

---

# Features

## Historical land cover
Primary source:
- local annual `GLC_FCS30D` rasters where available

Fallback:
- Dynamic World annual composite for recent years only

## Rainfall
- NASA GPM IMERG V07
- antecedent 1-day
- antecedent 3-day
- antecedent 7-day
- antecedent 30-day accumulated rainfall

## Dynamic soil moisture
- ERA5-Land
- 7-day mean soil moisture layer 1 (0–7 cm)
- 7-day mean soil moisture layer 2 (7–28 cm)

## Waterways
- current OSM / Geofabrik distance
- treated as a present-day contextual predictor with limitation

## Historical roads
- ohsome API workflow included
- disabled by default because large-scale requests can trigger rate limits
- activate only after the other Stage-05B features work

---

# Important rules

- No event date is invented.
- Undated landslide records stay available for static models but are excluded from event-time rainfall/soil-moisture analysis.
- Background controls receive the matched landslide's date.
- Current land cover is never substituted blindly for an old landslide year.
- Current roads are not treated as historical roads.


In [1]:
# OPTIONAL installation
# %pip install earthengine-api geopandas rasterio requests pyproj shapely pandas numpy matplotlib

from pathlib import Path
import re
import time
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
import ee

from pyproj import Transformer
from shapely.geometry import Point, shape

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


pandas: 3.0.2
geopandas: 1.1.4
rasterio: 1.5.0


## 1. Project paths and settings

In [2]:
# ============================================================
# CHANGE IF NEEDED
# ============================================================

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
EE_PROJECT = "fyp-landslide-malaysia"

CONTROL_RATIO = 2
RANDOM_SEED = 42

# Historical road extraction is intentionally OFF by default.
RUN_OHSOME_ROADS = False

# Optional ERA5-Land dynamic soil moisture.
RUN_ERA5_SOIL_MOISTURE = True


# ============================================================
# PATHS
# ============================================================

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_FINAL_PATH = (
    DEV_DIR
    / "03_derived_features"
    / "static_features_final"
    / "static_feature_samples_final.csv"
)

POSITIVE_META_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "malaysia_positive_inventory_ml_eligible.geojson"
)

LANDCOVER_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Landcover"
    / "GLC_FCS30D"
)

WATERWAYS_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Waterways"
)

OUTPUT_DIR = (
    DEV_DIR
    / "03_derived_features"
    / "temporal_features"
)

CACHE_DIR = OUTPUT_DIR / "cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Static final table": STATIC_FINAL_PATH,
    "ML-eligible positive metadata": POSITIVE_META_PATH,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\\n{path}")
    print("FOUND:", name, "->", path)

print("Land-cover root:", LANDCOVER_ROOT)
print("Waterways root:", WATERWAYS_ROOT)
print("Output:", OUTPUT_DIR)


FOUND: Static final table -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.csv
FOUND: ML-eligible positive metadata -> C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson
Land-cover root: C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Landcover\GLC_FCS30D
Waterways root: C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Waterways
Output: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features


## 2. Load static features and positive event metadata


In [3]:
static = pd.read_csv(STATIC_FINAL_PATH)

positive_meta = (
    gpd.read_file(POSITIVE_META_PATH)
    .to_crs("EPSG:4326")
)

positive_meta["event_date_parsed"] = pd.to_datetime(
    positive_meta.get("event_date"),
    errors="coerce"
)

positive_meta["event_year"] = (
    positive_meta["event_date_parsed"].dt.year
)

positive_meta["event_month"] = (
    positive_meta["event_date_parsed"].dt.month
)

positive_meta["event_day"] = (
    positive_meta["event_date_parsed"].dt.day
)

date_meta = (
    positive_meta[
        [
            "final_positive_id",
            "event_date_parsed",
            "event_year",
            "event_month",
            "event_day",
        ]
    ]
    .rename(
        columns={
            "final_positive_id":
                "sample_id"
        }
    )
)

static = static.merge(
    date_meta,
    on="sample_id",
    how="left"
)

print("Static table:", len(static))

print(
    "Landslides with event date:",
    int(
        (
            static["sample_class"].eq("landslide")
            & static["event_date_parsed"].notna()
        ).sum()
    )
)


Static table: 3200
Landslides with event date: 195


## 3. Define temporal-model eligible positives

For the temporal model, require:

- landslide positive
- final national core-static validity
- reliable event date
- year >= 2000

GPM IMERG V07 provides historical coverage from the TRMM/GPM era, so 2000+ is a practical temporal threshold.


In [4]:
temporal_positive_mask = (
    static["sample_class"].eq("landslide")
    & static["national_core_static_valid_final"].eq(True)
    & static["event_date_parsed"].notna()
    & static["event_year"].ge(2000)
)

temporal_positive = (
    static.loc[
        temporal_positive_mask
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Temporal-model eligible positives:",
    len(temporal_positive)
)

display(
    temporal_positive["event_year"]
    .value_counts()
    .sort_index()
    .rename_axis("event_year")
    .reset_index(name="records")
)


Temporal-model eligible positives: 189


,event_year,records
0,2007.0,2
1,2008.0,4
2,2009.0,7
3,2010.0,5
4,2011.0,14
5,2012.0,9
6,2013.0,16
7,2014.0,23
8,2015.0,15
9,2016.0,17


## 4. Select 2 date-matched controls per positive

Controls are sampled **without replacement** from static-valid background candidates.

Each control receives:
- `matched_positive_id`
- the same `reference_date`
- the same `reference_year`

The underlying background candidate location itself is unchanged.


In [5]:
valid_background = (
    static.loc[
        static["sample_class"].eq("background_candidate")
        & static["national_core_static_valid_final"].eq(True)
    ]
    .copy()
    .reset_index(drop=True)
)

required_controls = (
    len(temporal_positive)
    * CONTROL_RATIO
)

if len(valid_background) < required_controls:
    raise ValueError(
        "Not enough static-valid background candidates "
        "for the requested temporal control ratio."
    )

rng = np.random.default_rng(
    RANDOM_SEED
)

chosen_indices = rng.choice(
    valid_background.index.to_numpy(),
    size=required_controls,
    replace=False
)

selected_controls = (
    valid_background.loc[
        chosen_indices
    ]
    .copy()
    .reset_index(drop=True)
)

# Repeat each positive exactly CONTROL_RATIO times
match_reference = (
    temporal_positive[
        [
            "sample_id",
            "event_date_parsed",
            "event_year",
            "event_month",
            "event_day",
        ]
    ]
    .loc[
        temporal_positive.index.repeat(
            CONTROL_RATIO
        )
    ]
    .reset_index(drop=True)
)

selected_controls[
    "matched_positive_id"
] = match_reference[
    "sample_id"
].to_numpy()

selected_controls[
    "reference_date"
] = match_reference[
    "event_date_parsed"
].to_numpy()

selected_controls[
    "reference_year"
] = match_reference[
    "event_year"
].to_numpy()

selected_controls[
    "reference_month"
] = match_reference[
    "event_month"
].to_numpy()

selected_controls[
    "reference_day"
] = match_reference[
    "event_day"
].to_numpy()


temporal_positive[
    "matched_positive_id"
] = temporal_positive[
    "sample_id"
]

temporal_positive[
    "reference_date"
] = temporal_positive[
    "event_date_parsed"
]

temporal_positive[
    "reference_year"
] = temporal_positive[
    "event_year"
]

temporal_positive[
    "reference_month"
] = temporal_positive[
    "event_month"
]

temporal_positive[
    "reference_day"
] = temporal_positive[
    "event_day"
]


temporal_positive[
    "temporal_role"
] = "case"

selected_controls[
    "temporal_role"
] = "matched_control"


temporal_samples = pd.concat(
    [
        temporal_positive,
        selected_controls,
    ],
    ignore_index=True,
    sort=False
)

print(
    "Temporal cases:",
    int(
        temporal_samples[
            "temporal_role"
        ].eq("case").sum()
    )
)

print(
    "Temporal matched controls:",
    int(
        temporal_samples[
            "temporal_role"
        ].eq("matched_control").sum()
    )
)

print(
    "Total temporal samples:",
    len(temporal_samples)
)


Temporal cases: 189
Temporal matched controls: 378
Total temporal samples: 567


## 5. Save the temporal sampling design before feature extraction


In [6]:
temporal_design_path = (
    OUTPUT_DIR
    / "temporal_case_control_design.csv"
)

temporal_samples.to_csv(
    temporal_design_path,
    index=False
)

print("Saved:", temporal_design_path)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features\temporal_case_control_design.csv


# PART A — HISTORICAL LAND COVER

## 6. Discover local GLC_FCS30D annual rasters

The notebook searches recursively for `.tif` files containing a 4-digit year.

If Peninsular and East Malaysia are stored separately for the same year, both files are retained.

The point sampler tries all rasters for the requested year and returns the first valid pixel covering the point.


In [7]:
def build_year_raster_index(
    root
):
    index = {}

    if not root.exists():
        return index

    for path in root.rglob("*.tif"):

        matches = re.findall(
            r"(19\d{2}|20\d{2})",
            path.name
        )

        for match in matches:
            year = int(match)

            index.setdefault(
                year,
                []
            ).append(path)

    return index


landcover_index = build_year_raster_index(
    LANDCOVER_ROOT
)

print(
    "Local land-cover years found:"
)

print(
    sorted(
        landcover_index.keys()
    )
)


required_years = sorted(
    temporal_samples[
        "reference_year"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


print(
    "\nYears required by temporal samples:"
)

print(
    required_years
)


missing_local_years = [
    year
    for year in required_years
    if year not in landcover_index
]


print(
    "\nRequired years not found locally:"
)

print(
    missing_local_years
)


Local land-cover years found:
[2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]

Years required by temporal samples:
[2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2021, 2022]

Required years not found locally:
[2015, 2016, 2017, 2018, 2021, 2022]


## 7. Local annual raster point sampler


In [8]:
def sample_one_raster(
    raster_path,
    lon,
    lat
):
    try:

        with rasterio.open(
            raster_path
        ) as src:

            transformer = Transformer.from_crs(
                "EPSG:4326",
                src.crs,
                always_xy=True
            )

            x, y = transformer.transform(
                lon,
                lat
            )

            bounds = src.bounds

            if not (
                bounds.left <= x <= bounds.right
                and bounds.bottom <= y <= bounds.top
            ):
                return np.nan

            value = next(
                src.sample(
                    [
                        (
                            x,
                            y
                        )
                    ]
                )
            )[0]

            if not np.isfinite(
                value
            ):
                return np.nan

            if (
                src.nodata is not None
                and value == src.nodata
            ):
                return np.nan

            return float(
                value
            )

    except Exception:
        return np.nan


def sample_annual_landcover(
    year,
    lon,
    lat
):
    year = int(year)

    files = landcover_index.get(
        year,
        []
    )

    for path in files:

        value = sample_one_raster(
            path,
            lon,
            lat
        )

        if pd.notna(
            value
        ):
            return {
                "landcover_code_raw":
                    value,

                "landcover_source":
                    "GLC_FCS30D",

                "landcover_source_file":
                    path.name,

                "landcover_year":
                    year,

                "landcover_status":
                    "ok",
            }

    return {
        "landcover_code_raw":
            np.nan,

        "landcover_source":
            pd.NA,

        "landcover_source_file":
            pd.NA,

        "landcover_year":
            year,

        "landcover_status":
            "local_year_missing_or_no_valid_pixel",
    }


## 8. Connect to Earth Engine for fallback and climate extraction


In [9]:
try:

    ee.Initialize(
        project=EE_PROJECT
    )

    print(
        "Earth Engine initialized."
    )

except Exception:

    print(
        "Earth Engine authentication required."
    )

    ee.Authenticate()

    ee.Initialize(
        project=EE_PROJECT
    )

    print(
        "Earth Engine initialized."
    )


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized.


## 9. Dynamic World fallback

Dynamic World is used only when:
- the local annual historical raster is unavailable
- the requested year is covered by Dynamic World

The raw `label` is preserved.

No cross-product class harmonisation is performed automatically here.


In [10]:
DYNAMIC_WORLD_COLLECTION = (
    "GOOGLE/DYNAMICWORLD/V1"
)


def sample_dynamic_world_year(
    year,
    lon,
    lat
):
    year = int(year)

    point = ee.Geometry.Point(
        [
            float(lon),
            float(lat)
        ]
    )

    start = ee.Date.fromYMD(
        year,
        1,
        1
    )

    end = start.advance(
        1,
        "year"
    )

    collection = (
        ee.ImageCollection(
            DYNAMIC_WORLD_COLLECTION
        )
        .filterDate(
            start,
            end
        )
        .filterBounds(
            point
        )
    )

    count = collection.size().getInfo()

    if count == 0:
        return np.nan

    # Annual modal label
    annual_label = (
        collection
        .select(
            "label"
        )
        .mode()
    )

    result = annual_label.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=10,
        bestEffort=True
    ).getInfo()

    return result.get(
        "label"
    )


## 10. Extract historical land cover

The notebook tries:
1. local GLC_FCS30D
2. Dynamic World fallback

If both fail, the value stays missing.


In [11]:
landcover_rows = []

for i, row in temporal_samples.iterrows():

    local = sample_annual_landcover(
        row["reference_year"],
        row["longitude"],
        row["latitude"]
    )

    if (
        local["landcover_status"]
        == "ok"
    ):

        result = local

    else:

        dw_value = np.nan

        try:
            dw_value = sample_dynamic_world_year(
                row["reference_year"],
                row["longitude"],
                row["latitude"]
            )
        except Exception:
            dw_value = np.nan

        if pd.notna(
            dw_value
        ):

            result = {
                "landcover_code_raw":
                    float(
                        dw_value
                    ),

                "landcover_source":
                    "Dynamic_World",

                "landcover_source_file":
                    pd.NA,

                "landcover_year":
                    int(
                        row[
                            "reference_year"
                        ]
                    ),

                "landcover_status":
                    "ok_fallback_dynamic_world",
            }

        else:

            result = local

    result[
        "sample_id"
    ] = row[
        "sample_id"
    ]

    result[
        "matched_positive_id"
    ] = row[
        "matched_positive_id"
    ]

    landcover_rows.append(
        result
    )

    if (
        (i + 1)
        % 100
        == 0
    ):
        print(
            "Land cover:",
            i + 1,
            "/",
            len(
                temporal_samples
            )
        )


landcover_features = pd.DataFrame(
    landcover_rows
)


temporal_samples = temporal_samples.merge(
    landcover_features,
    on=[
        "sample_id",
        "matched_positive_id"
    ],
    how="left"
)


display(
    temporal_samples[
        [
            "sample_id",
            "reference_year",
            "landcover_source",
            "landcover_code_raw",
            "landcover_status"
        ]
    ].head(20)
)


Land cover: 100 / 567
Land cover: 200 / 567
Land cover: 300 / 567
Land cover: 400 / 567
Land cover: 500 / 567


,sample_id,reference_year,landcover_source,landcover_code_raw,landcover_status
0,POS_0001,2010.0,GLC_FCS30D,10.0,ok
1,POS_0002,2013.0,GLC_FCS30D,11.0,ok
2,POS_0003,2013.0,GLC_FCS30D,190.0,ok
3,POS_0006,2015.0,NaN,NaN,local_year_missing_or_no_valid_pixel
4,POS_0007,2011.0,GLC_FCS30D,190.0,ok
5,POS_0008,2011.0,GLC_FCS30D,190.0,ok
6,POS_0009,2018.0,Dynamic_World,6.0,ok_fallback_dynamic_world
7,POS_0010,2012.0,GLC_FCS30D,190.0,ok
8,POS_0013,2016.0,Dynamic_World,6.0,ok_fallback_dynamic_world
9,POS_0014,2015.0,Dynamic_World,6.0,ok_fallback_dynamic_world


# PART B — GPM ANTECEDENT RAINFALL

## 11. Rainfall method

Earth Engine collection:

`NASA/GPM_L3/IMERG_V07`

The `precipitation` band is a precipitation rate in **mm/hour** at 30-minute cadence.

Therefore each image contributes:

```text
precipitation × 0.5 hours
```

to accumulated rainfall.

The following windows are calculated **before the reference date**:

- 1 day
- 3 days
- 7 days
- 30 days

The event day itself is not included in the antecedent window.


In [12]:
GPM_COLLECTION = (
    "NASA/GPM_L3/IMERG_V07"
)

RAIN_WINDOWS_DAYS = [
    1,
    3,
    7,
    30,
]


def gpm_accumulation_image(
    reference_date,
    days
):
    end = ee.Date(
        pd.Timestamp(
            reference_date
        ).strftime(
            "%Y-%m-%d"
        )
    )

    start = end.advance(
        -int(days),
        "day"
    )

    collection = (
        ee.ImageCollection(
            GPM_COLLECTION
        )
        .filterDate(
            start,
            end
        )
        .select(
            "precipitation"
        )
    )

    # mm/hr * 0.5 hr per image
    return (
        collection
        .map(
            lambda img:
                img.multiply(
                    0.5
                )
        )
        .sum()
        .rename(
            f"rain_{days}d_mm"
        )
    )


## 12. Extract rainfall grouped by reference date

Grouping by date reduces repeated Earth Engine image construction.


In [13]:
def ee_sample_image_at_rows(
    image,
    group_df,
    property_names,
    scale
):
    features = []

    for _, row in group_df.iterrows():

        feature = ee.Feature(
            ee.Geometry.Point(
                [
                    float(
                        row[
                            "longitude"
                        ]
                    ),
                    float(
                        row[
                            "latitude"
                        ]
                    )
                ]
            ),
            {
                "temporal_row_id":
                    int(
                        row[
                            "temporal_row_id"
                        ]
                    )
            }
        )

        features.append(
            feature
        )

    fc = ee.FeatureCollection(
        features
    )

    result = image.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.first(),
        scale=scale
    ).getInfo()

    rows = []

    for feat in result.get(
        "features",
        []
    ):

        props = feat.get(
            "properties",
            {}
        )

        output = {
            "temporal_row_id":
                props.get(
                    "temporal_row_id"
                )
        }

        for name in property_names:
            output[name] = props.get(
                name
            )

        rows.append(
            output
        )

    return pd.DataFrame(
        rows
    )


temporal_samples = temporal_samples.reset_index(
    drop=True
)

temporal_samples[
    "temporal_row_id"
] = temporal_samples.index.astype(
    int
)


rain_parts = []

for reference_date, group in temporal_samples.groupby(
    "reference_date"
):

    images = [
        gpm_accumulation_image(
            reference_date,
            days
        )
        for days in RAIN_WINDOWS_DAYS
    ]

    combined = ee.Image.cat(
        images
    )

    names = [
        f"rain_{d}d_mm"
        for d in RAIN_WINDOWS_DAYS
    ]

    part = ee_sample_image_at_rows(
        combined,
        group,
        names,
        scale=11132
    )

    rain_parts.append(
        part
    )

    print(
        "Rainfall date complete:",
        pd.Timestamp(
            reference_date
        ).date(),
        "| rows:",
        len(group)
    )

    time.sleep(
        0.2
    )


rain_features = pd.concat(
    rain_parts,
    ignore_index=True
)


temporal_samples = temporal_samples.merge(
    rain_features,
    on="temporal_row_id",
    how="left"
)


Rainfall date complete: 2007-08-25 | rows: 3
Rainfall date complete: 2007-12-26 | rows: 3
Rainfall date complete: 2008-04-23 | rows: 3
Rainfall date complete: 2008-10-04 | rows: 3
Rainfall date complete: 2008-12-04 | rows: 3
Rainfall date complete: 2008-12-06 | rows: 3
Rainfall date complete: 2009-09-19 | rows: 6
Rainfall date complete: 2009-11-03 | rows: 6
Rainfall date complete: 2009-11-07 | rows: 3
Rainfall date complete: 2009-11-19 | rows: 3
Rainfall date complete: 2009-12-02 | rows: 3
Rainfall date complete: 2010-05-19 | rows: 3
Rainfall date complete: 2010-05-31 | rows: 3
Rainfall date complete: 2010-06-08 | rows: 3
Rainfall date complete: 2010-06-09 | rows: 3
Rainfall date complete: 2010-09-28 | rows: 3
Rainfall date complete: 2011-01-01 | rows: 3
Rainfall date complete: 2011-01-06 | rows: 3
Rainfall date complete: 2011-01-29 | rows: 3
Rainfall date complete: 2011-01-29 | rows: 3
Rainfall date complete: 2011-02-07 | rows: 3
Rainfall date complete: 2011-02-24 | rows: 3
Rainfall d

# PART C — ERA5-LAND SOIL MOISTURE

## 13. Optional 7-day soil-moisture context

ERA5-Land Daily Aggregated provides volumetric soil water for:
- layer 1: 0–7 cm
- layer 2: 7–28 cm

This is a dynamic hydrological state, not a replacement for SoilGrids texture.

Earth Engine source:
`ECMWF/ERA5_LAND/DAILY_AGGR`


In [14]:
ERA5_DAILY = (
    "ECMWF/ERA5_LAND/DAILY_AGGR"
)

ERA5_SOIL_BANDS = [
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
]


def era5_7day_soil_moisture_image(
    reference_date
):
    end = ee.Date(
        pd.Timestamp(
            reference_date
        ).strftime(
            "%Y-%m-%d"
        )
    )

    start = end.advance(
        -7,
        "day"
    )

    collection = (
        ee.ImageCollection(
            ERA5_DAILY
        )
        .filterDate(
            start,
            end
        )
        .select(
            ERA5_SOIL_BANDS
        )
    )

    return (
        collection
        .mean()
        .rename(
            [
                "soil_moisture_0_7cm_7d_mean",
                "soil_moisture_7_28cm_7d_mean",
            ]
        )
    )


In [15]:
if RUN_ERA5_SOIL_MOISTURE:

    soilmoist_parts = []

    for reference_date, group in temporal_samples.groupby(
        "reference_date"
    ):

        image = era5_7day_soil_moisture_image(
            reference_date
        )

        names = [
            "soil_moisture_0_7cm_7d_mean",
            "soil_moisture_7_28cm_7d_mean",
        ]

        part = ee_sample_image_at_rows(
            image,
            group,
            names,
            scale=11132
        )

        soilmoist_parts.append(
            part
        )

        print(
            "ERA5 soil moisture date complete:",
            pd.Timestamp(
                reference_date
            ).date()
        )

        time.sleep(
            0.2
        )

    soilmoist_features = pd.concat(
        soilmoist_parts,
        ignore_index=True
    )

    temporal_samples = temporal_samples.merge(
        soilmoist_features,
        on="temporal_row_id",
        how="left"
    )

else:

    temporal_samples[
        "soil_moisture_0_7cm_7d_mean"
    ] = np.nan

    temporal_samples[
        "soil_moisture_7_28cm_7d_mean"
    ] = np.nan


ERA5 soil moisture date complete: 2007-08-25
ERA5 soil moisture date complete: 2007-12-26
ERA5 soil moisture date complete: 2008-04-23
ERA5 soil moisture date complete: 2008-10-04
ERA5 soil moisture date complete: 2008-12-04
ERA5 soil moisture date complete: 2008-12-06
ERA5 soil moisture date complete: 2009-09-19
ERA5 soil moisture date complete: 2009-11-03
ERA5 soil moisture date complete: 2009-11-07
ERA5 soil moisture date complete: 2009-11-19
ERA5 soil moisture date complete: 2009-12-02
ERA5 soil moisture date complete: 2010-05-19
ERA5 soil moisture date complete: 2010-05-31
ERA5 soil moisture date complete: 2010-06-08
ERA5 soil moisture date complete: 2010-06-09
ERA5 soil moisture date complete: 2010-09-28
ERA5 soil moisture date complete: 2011-01-01
ERA5 soil moisture date complete: 2011-01-06
ERA5 soil moisture date complete: 2011-01-29
ERA5 soil moisture date complete: 2011-01-29
ERA5 soil moisture date complete: 2011-02-07
ERA5 soil moisture date complete: 2011-02-24
ERA5 soil 

# PART D — CURRENT WATERWAY DISTANCE

## 14. Discover local waterway vector files

This factor is treated as a present-day contextual predictor.

It is **not** claimed to reconstruct the exact drainage network at historical event time.


In [16]:
def discover_vector_files(
    root
):
    if not root.exists():
        return []

    extensions = {
        ".shp",
        ".gpkg",
        ".geojson",
    }

    return [
        p
        for p in root.rglob("*")
        if p.suffix.lower()
        in extensions
    ]


waterway_files = discover_vector_files(
    WATERWAYS_ROOT
)

print(
    "Waterway vector files found:"
)

for p in waterway_files:
    print(" -", p)


Waterway vector files found:
 - C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Waterways\gis_osm_waterways_free_1.shp


## 15. Load waterways and calculate distance

If several vector files exist, they are combined.

Only line-like geometries are used.


In [17]:
DISTANCE_CRS = "EPSG:3857"


if waterway_files:

    water_parts = []

    for path in waterway_files:

        try:
            gdf = gpd.read_file(
                path
            )

            if len(gdf) == 0:
                continue

            if gdf.crs is None:
                continue

            gdf = gdf.to_crs(
                "EPSG:4326"
            )

            line_mask = gdf.geometry.geom_type.isin(
                [
                    "LineString",
                    "MultiLineString"
                ]
            )

            gdf = gdf.loc[
                line_mask
            ].copy()

            if len(gdf):
                water_parts.append(
                    gdf[
                        [
                            "geometry"
                        ]
                    ]
                )

        except Exception as exc:
            print(
                "Skipped:",
                path,
                "|",
                exc
            )


    if water_parts:

        waterways = gpd.GeoDataFrame(
            pd.concat(
                water_parts,
                ignore_index=True
            ),
            geometry="geometry",
            crs="EPSG:4326"
        )

        waterways_m = waterways.to_crs(
            DISTANCE_CRS
        )

        water_union = (
            waterways_m.geometry.unary_union
        )

        temporal_points = gpd.GeoDataFrame(
            temporal_samples.copy(),
            geometry=gpd.points_from_xy(
                temporal_samples[
                    "longitude"
                ],
                temporal_samples[
                    "latitude"
                ]
            ),
            crs="EPSG:4326"
        ).to_crs(
            DISTANCE_CRS
        )

        temporal_samples[
            "distance_waterway_current_m"
        ] = temporal_points.geometry.apply(
            lambda p:
                float(
                    p.distance(
                        water_union
                    )
                )
        ).to_numpy()

        temporal_samples[
            "waterway_source"
        ] = "Current_Geofabrik_OSM"

    else:

        temporal_samples[
            "distance_waterway_current_m"
        ] = np.nan

        temporal_samples[
            "waterway_source"
        ] = pd.NA

else:

    temporal_samples[
        "distance_waterway_current_m"
    ] = np.nan

    temporal_samples[
        "waterway_source"
    ] = pd.NA

    print(
        "No local waterway vectors found. "
        "Waterway distance left missing."
    )


C:\Users\User\AppData\Local\Temp\ipykernel_17384\858209178.py:70: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  waterways_m.geometry.unary_union


# PART E — OPTIONAL HISTORICAL ROADS VIA OHSOME

## 16. ohsome configuration

The extraction uses OSM state at the sample's reference date.

Because hundreds of API calls can hit rate limits, this section is **disabled by default**.

When enabled:
- query roads in a local bounding box
- progressively expand search radius
- compute nearest historical OSM road
- cache every result

A missing road result is not interpreted as proof that no road existed; it may reflect incomplete historical OSM mapping.


In [18]:
OHSOME_URL = (
    "https://api.ohsome.org/v1/elements/geometry"
)

OHSOME_CACHE = (
    CACHE_DIR
    / "ohsome_road_distance_cache.csv"
)

ROAD_FILTER = (
    "type:way and highway=* "
    "and geometry:line"
)

ROAD_SEARCH_DEGREES = [
    0.02,
    0.05,
    0.10,
    0.20,
]


if OHSOME_CACHE.exists():
    road_cache = pd.read_csv(
        OHSOME_CACHE
    )
else:
    road_cache = pd.DataFrame(
        columns=[
            "sample_id",
            "reference_date",
            "distance_road_historical_m",
            "historical_road_feature_count",
            "historical_road_status",
        ]
    )


def query_ohsome_nearest_road(
    sample_id,
    lon,
    lat,
    reference_date
):

    date_string = pd.Timestamp(
        reference_date
    ).strftime(
        "%Y-%m-%d"
    )

    for delta in ROAD_SEARCH_DEGREES:

        bbox = ",".join(
            map(
                str,
                [
                    lon - delta,
                    lat - delta,
                    lon + delta,
                    lat + delta,
                ]
            )
        )

        payload = {
            "bboxes":
                bbox,

            "time":
                date_string,

            "filter":
                ROAD_FILTER,

            "properties":
                "tags",

            "clipGeometry":
                "false",
        }

        response = requests.post(
            OHSOME_URL,
            data=payload,
            timeout=120
        )

        if response.status_code == 429:
            return {
                "sample_id":
                    sample_id,

                "reference_date":
                    date_string,

                "distance_road_historical_m":
                    np.nan,

                "historical_road_feature_count":
                    np.nan,

                "historical_road_status":
                    "rate_limited",
            }

        response.raise_for_status()

        data = response.json()

        features = data.get(
            "features",
            []
        )

        if not features:
            time.sleep(
                0.4
            )
            continue

        roads = gpd.GeoDataFrame.from_features(
            features,
            crs="EPSG:4326"
        )

        line_mask = roads.geometry.geom_type.isin(
            [
                "LineString",
                "MultiLineString"
            ]
        )

        roads = roads.loc[
            line_mask
        ]

        if len(roads) == 0:
            continue

        point_gdf = gpd.GeoSeries(
            [
                Point(
                    lon,
                    lat
                )
            ],
            crs="EPSG:4326"
        ).to_crs(
            DISTANCE_CRS
        )

        roads_m = roads.to_crs(
            DISTANCE_CRS
        )

        distance = float(
            roads_m.distance(
                point_gdf.iloc[
                    0
                ]
            ).min()
        )

        return {
            "sample_id":
                sample_id,

            "reference_date":
                date_string,

            "distance_road_historical_m":
                distance,

            "historical_road_feature_count":
                int(
                    len(
                        roads
                    )
                ),

            "historical_road_status":
                "ok",
        }

    return {
        "sample_id":
            sample_id,

        "reference_date":
            date_string,

        "distance_road_historical_m":
            np.nan,

        "historical_road_feature_count":
            0,

        "historical_road_status":
            "no_osm_roads_returned",
    }


In [19]:
if RUN_OHSOME_ROADS:

    road_results = []

    cache_key = set(
        zip(
            road_cache[
                "sample_id"
            ].astype(str),

            road_cache[
                "reference_date"
            ].astype(str)
        )
    )

    for i, row in temporal_samples.iterrows():

        key = (
            str(
                row[
                    "sample_id"
                ]
            ),

            pd.Timestamp(
                row[
                    "reference_date"
                ]
            ).strftime(
                "%Y-%m-%d"
            )
        )

        if key in cache_key:
            continue

        result = query_ohsome_nearest_road(
            row[
                "sample_id"
            ],
            float(
                row[
                    "longitude"
                ]
            ),
            float(
                row[
                    "latitude"
                ]
            ),
            row[
                "reference_date"
            ]
        )

        road_results.append(
            result
        )

        if (
            result[
                "historical_road_status"
            ]
            == "rate_limited"
        ):
            print(
                "ohsome rate limit reached. "
                "Stop and rerun later; cache is preserved."
            )
            break

        if (
            (i + 1)
            % 25
            == 0
        ):
            print(
                "Historical roads:",
                i + 1,
                "/",
                len(
                    temporal_samples
                )
            )

        time.sleep(
            0.6
        )


    if road_results:

        new_cache = pd.DataFrame(
            road_results
        )

        road_cache = pd.concat(
            [
                road_cache,
                new_cache
            ],
            ignore_index=True
        ).drop_duplicates(
            [
                "sample_id",
                "reference_date"
            ],
            keep="last"
        )

        road_cache.to_csv(
            OHSOME_CACHE,
            index=False
        )


    temporal_samples[
        "_reference_date_string"
    ] = (
        pd.to_datetime(
            temporal_samples[
                "reference_date"
            ]
        )
        .dt.strftime(
            "%Y-%m-%d"
        )
    )

    temporal_samples = temporal_samples.merge(
        road_cache,
        left_on=[
            "sample_id",
            "_reference_date_string"
        ],
        right_on=[
            "sample_id",
            "reference_date"
        ],
        how="left",
        suffixes=(
            "",
            "_road_cache"
        )
    )

else:

    temporal_samples[
        "distance_road_historical_m"
    ] = np.nan

    temporal_samples[
        "historical_road_feature_count"
    ] = np.nan

    temporal_samples[
        "historical_road_status"
    ] = "not_run"


# PART F — STAGE-05B QC

## 17. Feature completeness summary


In [20]:
temporal_feature_cols = [
    "landcover_code_raw",
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
    "distance_waterway_current_m",
    "distance_road_historical_m",
]


summary_rows = []

for feature in temporal_feature_cols:

    if feature not in temporal_samples.columns:
        continue

    summary_rows.append({
        "feature":
            feature,

        "available_count":
            int(
                temporal_samples[
                    feature
                ].notna().sum()
            ),

        "missing_count":
            int(
                temporal_samples[
                    feature
                ].isna().sum()
            ),

        "available_pct":
            round(
                temporal_samples[
                    feature
                ].notna().mean()
                * 100,
                2
            ),
    })


temporal_feature_summary = pd.DataFrame(
    summary_rows
)

display(
    temporal_feature_summary
)

temporal_feature_summary.to_csv(
    OUTPUT_DIR
    / "temporal_feature_completeness_summary.csv",
    index=False
)


,feature,available_count,missing_count,available_pct
0,landcover_code_raw,503,64,88.71
1,rain_1d_mm,567,0,100.00
2,rain_3d_mm,567,0,100.00
3,rain_7d_mm,567,0,100.00
4,rain_30d_mm,567,0,100.00
5,soil_moisture_0_7cm_7d_mean,534,33,94.18
6,soil_moisture_7_28cm_7d_mean,534,33,94.18
7,distance_waterway_current_m,567,0,100.00
8,distance_road_historical_m,0,567,0.00


## 18. Land-cover source/year audit


In [21]:
landcover_audit = (
    temporal_samples.groupby(
        [
            "reference_year",
            "landcover_source",
            "landcover_status"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="records"
    )
    .sort_values(
        [
            "reference_year",
            "landcover_source"
        ]
    )
)

display(
    landcover_audit
)

landcover_audit.to_csv(
    OUTPUT_DIR
    / "landcover_source_year_audit.csv",
    index=False
)


,reference_year,landcover_source,landcover_status,records
0,2007.0,GLC_FCS30D,ok,6
1,2008.0,GLC_FCS30D,ok,12
2,2009.0,GLC_FCS30D,ok,21
3,2010.0,GLC_FCS30D,ok,15
4,2011.0,GLC_FCS30D,ok,42
5,2012.0,GLC_FCS30D,ok,27
6,2013.0,GLC_FCS30D,ok,48
7,2014.0,GLC_FCS30D,ok,69
8,2015.0,Dynamic_World,ok_fallback_dynamic_world,7
9,2015.0,NaN,local_year_missing_or_no_valid_pixel,38


## 19. Save Stage-05B temporal feature table


In [22]:
temporal_output = (
    OUTPUT_DIR
    / "temporal_feature_samples.csv"
)

temporal_samples.to_csv(
    temporal_output,
    index=False
)

print(
    "Saved:",
    temporal_output
)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features\temporal_feature_samples.csv


# Stage-05B interpretation

This table is a **temporal matched case-control dataset**, not yet the final model-ready table.

Before modelling:

1. review land-cover source coverage
2. confirm raw GLC_FCS30D class legend and harmonise categories
3. decide whether historical road coverage is sufficiently complete
4. evaluate rainfall / soil-moisture missingness
5. decide whether current waterway distance is acceptable in the enhanced model
6. create spatial validation groups

---

# Recommended model structure after Stage 05B

## Model A — National static baseline
Use static-valid samples:
- elevation
- slope
- aspect
- SoilGrids
- major-fault distance

## Model B — Geology-enhanced
Add structured geology where appropriate.

## Model C — Temporal / semi-dynamic
Use date-matched case-control samples and add:
- historical land cover
- antecedent rainfall
- ERA5-Land soil moisture
- historical road distance if reliable
- current waterway distance with explicit limitation

---

# Outputs to send for review

Please send:

```text
temporal_case_control_design.csv
temporal_feature_completeness_summary.csv
landcover_source_year_audit.csv
temporal_feature_samples.csv
```

If historical roads remain disabled, that is okay for the first Stage-05B review.
